## Automobile Price Prediction with XGBoost

Given *data about various automobiles*, let's try to predict the **price** of a given automobile.

We will use XGBoost to make our predictions.

Data source: https://www.kaggle.com/datasets/toramky/automobile-dataset

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import xgboost as xgb

from sklearn.metrics import r2_score

pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('Automobile_data.csv')
data

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2,164,audi,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950
4,2,164,audi,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,95,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,2952,ohc,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845
201,-1,95,volvo,gas,turbo,four,sedan,rwd,front,109.1,188.8,68.8,55.5,3049,ohc,four,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045
202,-1,95,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3012,ohcv,six,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485
203,-1,95,volvo,diesel,turbo,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3217,ohc,six,145,idi,3.01,3.4,23.0,106,4800,26,27,22470


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          205 non-null    int64  
 1   normalized-losses  205 non-null    str    
 2   make               205 non-null    str    
 3   fuel-type          205 non-null    str    
 4   aspiration         205 non-null    str    
 5   num-of-doors       205 non-null    str    
 6   body-style         205 non-null    str    
 7   drive-wheels       205 non-null    str    
 8   engine-location    205 non-null    str    
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    str    
 15  num-of-cylinders   205 non-null    str    
 16  engine-size        205 non-null    in

#### Cleaning

In [4]:
data = data.replace(-1, np.nan)
data = data.replace('?', np.nan)

In [5]:
data

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3.0,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
1,3.0,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,four,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
2,1.0,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,six,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
3,2.0,164,audi,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,four,109,mpfi,3.19,3.4,10.0,102,5500,24,30,13950
4,2.0,164,audi,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,five,136,mpfi,3.19,3.4,8.0,115,5500,18,22,17450
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,NaN,95,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,2952,ohc,four,141,mpfi,3.78,3.15,9.5,114,5400,23,28,16845
201,NaN,95,volvo,gas,turbo,four,sedan,rwd,front,109.1,188.8,68.8,55.5,3049,ohc,four,141,mpfi,3.78,3.15,8.7,160,5300,19,25,19045
202,NaN,95,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3012,ohcv,six,173,mpfi,3.58,2.87,8.8,134,5500,18,23,21485
203,NaN,95,volvo,diesel,turbo,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3217,ohc,six,145,idi,3.01,3.4,23.0,106,4800,26,27,22470


In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          183 non-null    float64
 1   normalized-losses  164 non-null    str    
 2   make               205 non-null    str    
 3   fuel-type          205 non-null    str    
 4   aspiration         205 non-null    str    
 5   num-of-doors       203 non-null    str    
 6   body-style         205 non-null    str    
 7   drive-wheels       205 non-null    str    
 8   engine-location    205 non-null    str    
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    str    
 15  num-of-cylinders   205 non-null    str    
 16  engine-size        205 non-null    in

In [7]:
for column in ['normalized-losses', 'bore', 'stroke', 'horsepower', 'peak-rpm', 'price']:
    data[column] = data[column].astype(float)

In [8]:
data.isna().sum()

symboling            22
normalized-losses    41
make                  0
fuel-type             0
aspiration            0
num-of-doors          2
body-style            0
drive-wheels          0
engine-location       0
wheel-base            0
length                0
width                 0
height                0
curb-weight           0
engine-type           0
num-of-cylinders      0
engine-size           0
fuel-system           0
bore                  4
stroke                4
compression-ratio     0
horsepower            2
peak-rpm              2
city-mpg              0
highway-mpg           0
price                 4
dtype: int64

In [9]:
for column in data.columns:
    if data.dtypes[column] != 'str' and data.isna().sum()[column] > 0:  # For all numerical columns with missing values
        data[column] = data[column].fillna(data[column].mean()) #  Fill missing values with the column's mean

In [10]:
data.isna().sum()

symboling            0
normalized-losses    0
make                 0
fuel-type            0
aspiration           0
num-of-doors         2
body-style           0
drive-wheels         0
engine-location      0
wheel-base           0
length               0
width                0
height               0
curb-weight          0
engine-type          0
num-of-cylinders     0
engine-size          0
fuel-system          0
bore                 0
stroke               0
compression-ratio    0
horsepower           0
peak-rpm             0
city-mpg             0
highway-mpg          0
price                0
dtype: int64

#### Encoding ordinal features

In [11]:
{column: list(data[column].unique()) for column in data.columns if data.dtypes[column] == 'str'}

{'make': ['alfa-romero',
  'audi',
  'bmw',
  'chevrolet',
  'dodge',
  'honda',
  'isuzu',
  'jaguar',
  'mazda',
  'mercedes-benz',
  'mercury',
  'mitsubishi',
  'nissan',
  'peugot',
  'plymouth',
  'porsche',
  'renault',
  'saab',
  'subaru',
  'toyota',
  'volkswagen',
  'volvo'],
 'fuel-type': ['gas', 'diesel'],
 'aspiration': ['std', 'turbo'],
 'num-of-doors': ['two', 'four', nan],
 'body-style': ['convertible', 'hatchback', 'sedan', 'wagon', 'hardtop'],
 'drive-wheels': ['rwd', 'fwd', '4wd'],
 'engine-location': ['front', 'rear'],
 'engine-type': ['dohc', 'ohcv', 'ohc', 'l', 'rotor', 'ohcf', 'dohcv'],
 'num-of-cylinders': ['four',
  'six',
  'five',
  'three',
  'twelve',
  'two',
  'eight'],
 'fuel-system': ['mpfi', '2bbl', 'mfi', '1bbl', 'spfi', '4bbl', 'idi', 'spdi']}

In [12]:
num_cyl_ordering = {
    'two': 2,
    'three': 3,
    'four': 4,
    'five': 5,
    'six': 6,
    'eight': 8,
    'twelve': 12
}

data['num-of-cylinders'] = data['num-of-cylinders'].replace(num_cyl_ordering)

In [13]:
data

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,width,height,curb-weight,engine-type,num-of-cylinders,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3.000000,122.0,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,4,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3.000000,122.0,alfa-romero,gas,std,two,convertible,rwd,front,88.6,168.8,64.1,48.8,2548,dohc,4,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1.000000,122.0,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,171.2,65.5,52.4,2823,ohcv,6,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2.000000,164.0,audi,gas,std,four,sedan,fwd,front,99.8,176.6,66.2,54.3,2337,ohc,4,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2.000000,164.0,audi,gas,std,four,sedan,4wd,front,99.4,176.6,66.4,54.3,2824,ohc,5,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,1.054645,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,2952,ohc,4,141,mpfi,3.78,3.15,9.5,114.0,5400.0,23,28,16845.0
201,1.054645,95.0,volvo,gas,turbo,four,sedan,rwd,front,109.1,188.8,68.8,55.5,3049,ohc,4,141,mpfi,3.78,3.15,8.7,160.0,5300.0,19,25,19045.0
202,1.054645,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3012,ohcv,6,173,mpfi,3.58,2.87,8.8,134.0,5500.0,18,23,21485.0
203,1.054645,95.0,volvo,diesel,turbo,four,sedan,rwd,front,109.1,188.8,68.9,55.5,3217,ohc,6,145,idi,3.01,3.40,23.0,106.0,4800.0,26,27,22470.0


In [14]:
data['num-of-doors'].mode()

0    four
Name: num-of-doors, dtype: str

In [15]:
# Filling remaining missing values
data['num-of-doors'] = data['num-of-doors'].apply(lambda x: 0 if x == 'two' else 1)

#### Filling remaining missing values

In [16]:
print("Total missing values:", data.isna().sum().sum())

Total missing values: 0


#### Encoding

In [17]:
def binary_encode(df, columns, positive_values):
    df = df.copy()
    for column, positive_value in zip(columns, positive_values):
        df[column] = df[column].apply(lambda x: 1 if x == positive_value else 0)
    return df

In [18]:
{column: list(data[column].unique()) for column in data.columns if data.dtypes[column] == 'str'}

{'make': ['alfa-romero',
  'audi',
  'bmw',
  'chevrolet',
  'dodge',
  'honda',
  'isuzu',
  'jaguar',
  'mazda',
  'mercedes-benz',
  'mercury',
  'mitsubishi',
  'nissan',
  'peugot',
  'plymouth',
  'porsche',
  'renault',
  'saab',
  'subaru',
  'toyota',
  'volkswagen',
  'volvo'],
 'fuel-type': ['gas', 'diesel'],
 'aspiration': ['std', 'turbo'],
 'body-style': ['convertible', 'hatchback', 'sedan', 'wagon', 'hardtop'],
 'drive-wheels': ['rwd', 'fwd', '4wd'],
 'engine-location': ['front', 'rear'],
 'engine-type': ['dohc', 'ohcv', 'ohc', 'l', 'rotor', 'ohcf', 'dohcv'],
 'fuel-system': ['mpfi', '2bbl', 'mfi', '1bbl', 'spfi', '4bbl', 'idi', 'spdi']}

In [19]:
binary_features = ['fuel-type', 'aspiration', 'engine-location']

binary_positive_values = ['diesel', 'turbo', 'front']

data = binary_encode(
    data,
    binary_features,
    binary_positive_values
)

In [20]:
def onehot_encode(df, columns, prefixes):
    df = df.copy()
    for column, prefix in zip(columns, prefixes):
        dummies = pd.get_dummies(df[column], dtype=int, prefix=prefix)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    return df

In [21]:
nominal_features = ['make', 'drive-wheels', 'engine-type', 'fuel-system', 'body-style']

nominal_prefixes = ['m', 'dw', 'et', 'fs', 'bs']

data = onehot_encode(
    data,
    nominal_features, 
    nominal_prefixes
)

In [22]:
data

,symboling,normalized-losses,fuel-type,aspiration,num-of-doors,engine-location,wheel-base,length,width,height,curb-weight,num-of-cylinders,engine-size,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price,m_alfa-romero,m_audi,m_bmw,m_chevrolet,m_dodge,m_honda,m_isuzu,m_jaguar,m_mazda,m_mercedes-benz,m_mercury,m_mitsubishi,m_nissan,m_peugot,m_plymouth,m_porsche,m_renault,m_saab,m_subaru,m_toyota,m_volkswagen,m_volvo,dw_4wd,dw_fwd,dw_rwd,et_dohc,et_dohcv,et_l,et_ohc,et_ohcf,et_ohcv,et_rotor,fs_1bbl,fs_2bbl,fs_4bbl,fs_idi,fs_mfi,fs_mpfi,fs_spdi,fs_spfi,bs_convertible,bs_hardtop,bs_hatchback,bs_sedan,bs_wagon
0,3.000000,122.0,0,0,0,1,88.6,168.8,64.1,48.8,2548,4,130,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0
1,3.000000,122.0,0,0,0,1,88.6,168.8,64.1,48.8,2548,4,130,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0
2,1.000000,122.0,0,0,0,1,94.5,171.2,65.5,52.4,2823,6,152,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0
3,2.000000,164.0,0,0,1,1,99.8,176.6,66.2,54.3,2337,4,109,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
4,2.000000,164.0,0,0,1,1,99.4,176.6,66.4,54.3,2824,5,136,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,1.054645,95.0,0,0,1,1,109.1,188.8,68.9,55.5,2952,4,141,3.78,3.15,9.5,114.0,5400.0,23,28,16845.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
201,1.054645,95.0,0,1,1,1,109.1,188.8,68.8,55.5,3049,4,141,3.78,3.15,8.7,160.0,5300.0,19,25,19045.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0
202,1.054645,95.0,0,0,1,1,109.1,188.8,68.9,55.5,3012,6,173,3.58,2.87,8.8,134.0,5500.0,18,23,21485.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0
203,1.054645,95.0,1,1,1,1,109.1,188.8,68.9,55.5,3217,6,145,3.01,3.40,23.0,106.0,4800.0,26,27,22470.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0


In [23]:
print("Total non-numeric columns:", (data.dtypes=='str').sum())

Total non-numeric columns: 0


#### Splitting/Scaling

In [24]:
y = data['price'].copy()
X = data.drop('price', axis=1).copy()

In [25]:
scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=20)

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=20)

In [27]:
X_train

,symboling,normalized-losses,fuel-type,aspiration,num-of-doors,engine-location,wheel-base,length,width,height,curb-weight,num-of-cylinders,engine-size,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,m_alfa-romero,m_audi,m_bmw,m_chevrolet,m_dodge,m_honda,m_isuzu,m_jaguar,m_mazda,m_mercedes-benz,m_mercury,m_mitsubishi,m_nissan,m_peugot,m_plymouth,m_porsche,m_renault,m_saab,m_subaru,m_toyota,m_volkswagen,m_volvo,dw_4wd,dw_fwd,dw_rwd,et_dohc,et_dohcv,et_l,et_ohc,et_ohcf,et_ohcv,et_rotor,fs_1bbl,fs_2bbl,fs_4bbl,fs_idi,fs_mfi,fs_mpfi,fs_spdi,fs_spfi,bs_convertible,bs_hardtop,bs_hatchback,bs_sedan,bs_wagon
127,1.822954e+00,0.000000,-0.328798,-0.469295,-1.141653,-8.205689,-1.540948,-0.418396,-0.424215,-0.871725,0.385889,1.502032,1.615095,1.518415,-1.136149,-0.162161,2.606210,1.628010,-1.259470,-0.837195,-0.121867,-0.188025,-0.201517,-0.121867,-0.214286,-0.260208,-0.141069,-0.121867,-0.300708,-0.201517,-0.070014,-0.260208,-0.310253,-0.23812,-0.188025,6.324555,-0.099258,-0.17364,-0.249351,-0.430083,-0.249351,-0.23812,-0.214286,-1.188177,1.302831,-0.249351,-0.070014,-0.249351,-1.611363,3.559026,-0.260208,-0.141069,-0.238120,-0.689072,-0.121867,-0.328798,-0.070014,1.086670,-0.214286,-0.070014,-0.173640,4.962358,-0.720082,-0.938474,-0.372678
177,2.080736e-16,-1.803590,-0.328798,-0.469295,0.875923,0.121867,0.606521,0.126002,0.276731,0.071843,-0.187840,-0.352887,-0.118132,-0.073103,0.909683,-0.364062,-0.310891,-1.944812,0.272823,0.181783,-0.121867,-0.188025,-0.201517,-0.121867,-0.214286,-0.260208,-0.141069,-0.121867,-0.300708,-0.201517,-0.070014,-0.260208,-0.310253,-0.23812,-0.188025,-0.158114,-0.099258,-0.17364,-0.249351,2.325134,-0.249351,-0.23812,-0.214286,0.841625,-0.767559,-0.249351,-0.070014,-0.249351,0.620593,-0.280976,-0.260208,-0.141069,-0.238120,-0.689072,-0.121867,-0.328798,-0.070014,1.086670,-0.214286,-0.070014,-0.173640,-0.201517,1.388730,-0.938474,-0.372678
0,1.822954e+00,0.000000,-0.328798,-0.469295,-1.141653,0.121867,-1.690772,-0.426521,-0.844782,-2.020417,-0.014566,-0.352887,0.074449,0.519089,-1.839404,-0.288349,0.171065,-0.263484,-0.646553,-0.546059,8.205689,-0.188025,-0.201517,-0.121867,-0.214286,-0.260208,-0.141069,-0.121867,-0.300708,-0.201517,-0.070014,-0.260208,-0.310253,-0.23812,-0.188025,-0.158114,-0.099258,-0.17364,-0.249351,-0.430083,-0.249351,-0.23812,-0.214286,-1.188177,1.302831,4.010403,-0.070014,-0.249351,-1.611363,-0.280976,-0.260208,-0.141069,-0.238120,-0.689072,-0.121867,-0.328798,-0.070014,1.086670,-0.214286,-0.070014,5.759051,-0.201517,-0.720082,-0.938474,-0.372678
40,-9.882866e-01,-1.170751,-0.328798,-0.469295,0.875923,0.121867,-0.375655,0.109752,-1.592458,0.153893,-0.353413,-0.352887,-0.407003,-0.665296,1.037547,-0.288349,-0.463087,1.417844,0.272823,0.327351,-0.121867,-0.188025,-0.201517,-0.121867,-0.214286,3.843076,-0.141069,-0.121867,-0.300708,-0.201517,-0.070014,-0.260208,-0.310253,-0.23812,-0.188025,-0.158114,-0.099258,-0.17364,-0.249351,-0.430083,-0.249351,-0.23812,-0.214286,0.841625,-0.767559,-0.249351,-0.070014,-0.249351,0.620593,-0.280976,-0.260208,-0.141069,4.199567,-0.689072,-0.121867,-0.328798,-0.070014,-0.920243,-0.214286,-0.070014,-0.173640,-0.201517,-0.720082,1.065559,-0.372678
6,-5.120656e-02,1.139109,-0.328798,-0.469295,0.875923,0.121867,1.172521,1.515437,2.566488,0.810288,0.555313,0.574572,0.218885,-0.517248,0.462157,-0.414537,0.145699,0.787346,-0.953012,-0.837195,-0.121867,5.318432,-0.201517,-0.121867,-0.214286,-0.260208,-0.141069,-0.121867,-0.300708,-0.201517,-0.070014,-0.260208,-0.310253,-0.23812,-0.188025,-0.158114,-0.099258,-0.17364,-0.249351,-0.430083,-0.249351,-0.23812,-0.214286,0.841625,-0.767559,-0.249351,-0.070014,-0.249351,0.620593,-0.280976,-0.260208,-0.141069,-0.238120,-0.689072,-0.121867,-0.328798,-0.070014,1.086670,-0.214286,-0.070014,-0.173640,-0.201517,-0.720082,1.065559,-0.372678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [28]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

### Training

In [ ]:
params = {'learning_rate': 0.001, 'max_depth': 6}

model = xgb.train(params, dtrain, evals=[(dval, 'eval')], num_boost_round=10000, early_stopping_rounds=10)

[0]	eval-rmse:10637.36224
[1]	eval-rmse:10629.64647
[2]	eval-rmse:10621.93815
[3]	eval-rmse:10614.23738
[4]	eval-rmse:10606.54423
[5]	eval-rmse:10598.85931
[6]	eval-rmse:10591.18107
[7]	eval-rmse:10583.51133
[8]	eval-rmse:10575.84855
[9]	eval-rmse:10568.19359
[10]	eval-rmse:10560.54579
[11]	eval-rmse:10552.90619
[12]	eval-rmse:10545.27402
[13]	eval-rmse:10537.64937
[14]	eval-rmse:10530.03236
[15]	eval-rmse:10522.42289
[16]	eval-rmse:10514.82087
[17]	eval-rmse:10507.22653
[18]	eval-rmse:10499.63956
[19]	eval-rmse:10492.06025
[20]	eval-rmse:10484.48859
[21]	eval-rmse:10476.92444
[22]	eval-rmse:10469.36760
[23]	eval-rmse:10461.64207
[24]	eval-rmse:10454.10019
[25]	eval-rmse:10446.56546
[26]	eval-rmse:10438.86261
[27]	eval-rmse:10431.34285
[28]	eval-rmse:10423.83111
[29]	eval-rmse:10416.15041
[30]	eval-rmse:10408.65385
[31]	eval-rmse:10401.16384
[32]	eval-rmse:10393.50618
[33]	eval-rmse:10386.03113
[34]	eval-rmse:10378.56363
[35]	eval-rmse:10370.92855
[36]	eval-rmse:10363.47539
[37]	eval-r

### Results

In [30]:
y_true = np.array(y_test, dtype=float)
y_pred = np.array(model.predict(dtest), dtype=float)

In [31]:
print("R^2 Score: {:.4f}".format(r2_score(y_true, y_pred)))

R^2 Score: 0.2783
